<a href="https://colab.research.google.com/github/lsgrep/agents/blob/claude/agent-building-lessons-16749f/notebooks/10_going_live.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 10 — Going live

**The claim you should be able to make when you finish:** *"I predicted on
paper, then measured against a real model, and I can explain the gap. Within 2x
means my model of the system works; off by 10x means a term is missing."*

Every other lab in this ladder runs offline, deterministically, for free. That
was deliberate, not a compromise: the loop, the accounting, the failure modes and
the eval harness are all **your code**, and testing your code against a paid,
slow, non-deterministic API teaches you less, slower, because when something
breaks you cannot tell whether it was you or the sampler.

This lab is the seam. One line changes, and the same lessons run against Claude.

**This lab spends money.** Not much — a few cents at the sizes here — but it is
the only one that does, and every cell that costs anything is marked. Cells
without a key set will skip themselves rather than fail.

In [ ]:
# Cell 1 — bootstrap. This is the one lab that wants an API key.
REPO, BRANCH = "https://github.com/lsgrep/agents.git", "claude/agent-building-lessons-16749f"

import os, subprocess, sys

if not os.path.isdir("agents"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "agents", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("agents"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'anthropic', 'matplotlib'], check=True)

import agentlab
env = agentlab.notebook_setup()

## 0. Do you have what this needs?

Set `ANTHROPIC_API_KEY` in your environment (in Colab: the key icon in the left
sidebar, then "Notebook access"). Without it, everything below skips and the
markdown still tells you what it would have shown.

In [ ]:
from agentlab.env import detect

env = detect()
LIVE = env.can_run_live
print(f"sdk: {env.has_anthropic_sdk}   key: {env.has_api_key}   -> live cells will "
      f"{'RUN (and spend)' if LIVE else 'skip'}")

## 1. Check the token estimate

`budget.estimate_tokens` divides characters by 3.8. It is a planning heuristic
and it is wrong — the question is by how much, for *your* text, so you know how
far to trust every projection in labs 2 and 8.

Do this once. Then plan with the estimate, knowing its bias.

In [ ]:
# requires: live
from agentlab.budget import estimate_tokens

SYSTEM = """You are an operations assistant for a warehouse system.
Answer from tool results only. If a tool fails, say what you tried and stop.
Never place an order without being asked to."""

TOOLS = [
    {"name": "lookup_stock",
     "description": "Return the number of units of one product currently in stock.",
     "input_schema": {"type": "object",
                      "properties": {"product": {"type": "string", "description": "Product name."}},
                      "required": ["product"]}},
]

if LIVE:
    from agentlab.providers import count_tokens

    real = count_tokens([{"role": "user", "content": "How many widgets are in stock?"}],
                        system=SYSTEM, tools=TOOLS)
    guess = estimate_tokens(SYSTEM) + sum(estimate_tokens(str(t)) for t in TOOLS) + estimate_tokens(
        "How many widgets are in stock?")
    print(f"estimated: {guess:>6} tokens")
    print(f"actual:    {real:>6} tokens")
    print(f"ratio:     {real / guess:>6.2f}x")
    print("\nJSON and schemas are denser than prose, so the estimate usually runs low.")
    print("Scale your projections by this ratio and they are good enough to plan with.")
else:
    print("skipped — no key")

## 2. Run the loop, for real

The only change from lab 1 is the model object. Everything else — the registry,
the invariants, the trace — is identical.

In [ ]:
# requires: live
from agentlab.loop import Tool, ToolRegistry, run

STOCK = {"widget": 12, "sprocket": 7, "gizmo": 30}
registry = ToolRegistry([
    Tool("lookup_stock", "Return the number of units of one product currently in stock.",
         {"type": "object", "properties": {"product": {"type": "string"}}, "required": ["product"]},
         fn=lambda product: STOCK.get(product, 0)),
])

if LIVE:
    from agentlab.providers import claude

    model = claude(model="claude-opus-5", cache=True)
    trace = run(model, registry, "How many widgets and gizmos do we have in total?",
                system=SYSTEM, max_steps=8)

    print(trace)
    print()
    for call in trace.calls:
        print(f"  {call.name}({call.input}) -> {call.result}")
    print(f"\n{trace.final_text}")
    print(f"\nspend: {model.spend()}")
else:
    print("skipped — no key")

Two details in `providers.claude` worth copying rather than skimming, because
both are quiet failures rather than errors:

**Thinking blocks come back in `content` and must go back unchanged.** That is
why the provider returns the whole content list and `loop.run` appends it
verbatim. Extracting just the text is the single most common way to break an
agent loop subtly — it works, it just gets worse.

**Cache breakpoints go at the end of stable regions.** Render order is `tools`
-> `system` -> `messages`, so the provider puts one at the end of the tool list,
one at the end of the system prompt, and one at the end of the conversation so
far.

## 3. The measurement that catches a silent invalidator

Lab 2 said: assume nothing about your cache hit rate, measure it. Here is the
measurement, and it is the single most valuable cell in this lab.

`cache_read_input_tokens` at zero across repeated calls means you are paying
full price on every turn while your code looks completely correct.

In [ ]:
# requires: live
if LIVE:
    model = claude(model="claude-opus-5", cache=True)
    trace = run(model, registry, "How many widgets, sprockets and gizmos do we have?",
                system=SYSTEM, max_steps=8)

    print(f"{'turn':>5} {'input':>8} {'cache read':>11} {'cache write':>12} {'output':>8}")
    for i, r in enumerate(model.responses, 1):
        u = r.usage
        print(f"{i:>5} {u.input_tokens:>8} "
              f"{getattr(u, 'cache_read_input_tokens', 0) or 0:>11} "
              f"{getattr(u, 'cache_creation_input_tokens', 0) or 0:>12} "
              f"{u.output_tokens:>8}")

    hit = model.cache_hit_rate()
    print(f"\ncache hit rate: {hit:.1%}" if hit is not None else "no data")
    print("Turn 1 writes. Every turn after should show a large cache read.")
    print("If those reads are zero, find the invalidator before tuning anything else.")
else:
    print("skipped — no key")

### The invalidator, demonstrated

Put something that changes every turn into the system prompt and watch the
saving vanish. This is worth running once so you recognise the signature.

In [ ]:
# requires: live
if LIVE:
    import datetime

    def run_with(system_fn, label):
        m = claude(model="claude-opus-5", cache=True)
        run(m, registry, "How many widgets, sprockets and gizmos do we have?",
            system=system_fn(), max_steps=8)
        reads = sum(getattr(r.usage, "cache_read_input_tokens", 0) or 0 for r in m.responses)
        print(f"{label:<34} cache reads: {reads:>7,}   spend: ${m.spend()['usd']:.4f}")

    run_with(lambda: SYSTEM, "stable system prompt")
    run_with(lambda: SYSTEM + f"\nCurrent time: {datetime.datetime.now()}", "with a timestamp in it")
    print("\nThe timestamp changes the prefix, so nothing after it can be reused.")
    print("A per-request id, an unsorted tool list, or a mid-session model switch")
    print("all do exactly the same thing, just less obviously.")
else:
    print("skipped — no key")

## 4. Measure your own `p_step`

Lab 3 asked you to use a per-step success rate from your own traces rather than
a benchmark number. Here is how you get one: run a set of tasks, count the steps
that made progress.

Twenty tasks is not enough to establish a rate precisely — lab 6 is very clear
about that — but it is enough to know whether you are at 0.99 or 0.9, and those
two lead to completely different designs.

In [ ]:
# requires: live
from agentlab.evals import Case, evaluate
from agentlab import reliability as rel

CASES = [
    Case("c1", "How many widgets are in stock?", expect=r"12", requires=("lookup_stock",)),
    Case("c2", "Do we have more gizmos than sprockets?", expect=r"(gizmo|yes|more)",
         requires=("lookup_stock",)),
    Case("c3", "What's the total across all three products?", expect=r"49",
         requires=("lookup_stock",)),
]

if LIVE:
    report = evaluate(CASES, lambda c: run(claude(), registry, c.prompt, system=SYSTEM, max_steps=8))
    print(report)
    print()
    steps = sum(t.steps for t in report.traces)
    errors = sum(t.tool_errors for t in report.traces)
    p_step = 1 - errors / max(1, sum(t.tool_calls for t in report.traces))
    print(f"{steps} steps across {len(CASES)} tasks, {errors} tool errors")
    print(f"crude p_step: {p_step:.1%}  (tool-level only — it does not count wrong-but-valid calls)")
    print()
    print(rel.derive_horizon(p_step=min(p_step, 0.999), n_steps=30))
else:
    print("skipped — no key")

Be honest about what that number is. Counting tool *errors* undercounts, because
the expensive failures are calls that succeed and are wrong — the right tool with
the wrong argument, the right search with the wrong query. Getting the real
`p_step` needs per-step labels, which is a genuine cost.

A reasonable middle path: label per-step correctness on 20 runs by hand, once,
and calibrate the cheap automated proxy against it. Same shape as calibrating a
judge in lab 6.

## 5. Effort, and where it actually pays

`output_config: {effort: ...}` trades thoroughness against tokens. Lower effort
means fewer, more consolidated tool calls and less preamble.

The interesting measurement is not tokens per request — it is **cost per
completed task**. A cheaper request that needs more turns is not cheaper, and
that is exactly the trap lab 8's fan-out sweep set as well.

In [ ]:
# requires: live
if LIVE:
    print(f"{'effort':>8} {'steps':>7} {'calls':>7} {'$':>9} {'passed':>8}")
    for effort in ("low", "medium", "high"):
        m = claude(model="claude-opus-5", effort=effort)
        report = evaluate(CASES, lambda c: run(m, registry, c.prompt, system=SYSTEM, max_steps=8))
        spend = m.spend()
        passed = report.passed
        cost_per_success = spend["usd"] / passed if passed else float("inf")
        print(f"{effort:>8} {sum(t.steps for t in report.traces):>7} "
              f"{sum(t.tool_calls for t in report.traces):>7} {spend['usd']:>9.4f} "
              f"{passed}/{report.n:>6}")
    print("\nOn a task this easy, low effort should win outright. That is the finding:")
    print("high effort is not a quality setting you leave on, it is a lever you tune")
    print("per route, and easy routes do not repay it.")
else:
    print("skipped — no key")

## 6. Predict, then measure, then explain the gap

The habit this whole ladder is built around. Predict the cost of a run from the
formula, then run it and compare.

In [ ]:
# requires: live
from agentlab.budget import run_cost

if LIVE:
    m = claude(model="claude-opus-5", cache=True)
    trace = run(m, registry, "What's the total stock across all three products?",
                system=SYSTEM, max_steps=8)

    predicted = run_cost(trace.shape(), cached=True)
    actual = m.spend()

    print(f"predicted: ${predicted.usd:.5f}")
    print(f"actual:    ${actual['usd']:.5f}")
    ratio = actual["usd"] / predicted.usd if predicted.usd else float("inf")
    print(f"ratio:     {ratio:.2f}x")
    print()
    if 0.5 <= ratio <= 2.0:
        print("Within 2x. Your mental model of the system works.")
    else:
        print("Off by more than 2x — a term is missing. Usual suspects:")
        print("  - thinking tokens, which bill as output and are easy to forget")
        print("  - the token estimate's bias (section 1)")
        print("  - cache writes on a run too short to reuse them")
        print("Finding which one is the lesson.")
else:
    print("skipped — no key")

## 7. What to take back to your own system

In rough order of what it will buy you:

1. **Measure your cache hit rate today.** It is one line, and if it is low the
   fix is usually a one-line change with an 80% saving behind it.
2. **Count your tool schemas.** With the real tokenizer. That number is charged
   on every request of every run, forever.
3. **Count the steps in a median run** and put them through
   `reliability.derive_horizon`. If the answer is uncomfortable, the design
   needs to change, not the prompt.
4. **Add `requires` and `forbids` to your eval cases.** No judge, no labels, and
   it catches the failure class that outcome scoring cannot see.
5. **Print your gate's blind spot** next to its verdict. Every time.
6. **Run `security.analyze()` on your tool surface** and put it in CI.
7. **Log `repeat_calls` and `stop_reason`** per run. The `max_steps` rate climbs
   before your pass rate falls.

## What you can now say

- *"We predicted the run cost on paper and landed within 2x of the bill."*
- *"Our cache hit rate is measured from `cache_read_input_tokens`, and we know
  what invalidates it."*
- *"Our `p_step` came from our own traces — and we know that counting tool
  errors undercounts it."*
- *"Effort is tuned per route. Easy routes don't repay high effort, and cost per
  completed task is the metric."*

---

That is the ladder. [`docs/INTERVIEW_MAP.md`](../docs/INTERVIEW_MAP.md) maps
every claim above to the lab that produces its receipt — and lists, plainly,
what these labs do not cover.